In [ ]:
from google.colab import files
uploaded = files.upload()

Saving 1_f.csv - 1_f.csv.csv to 1_f.csv - 1_f.csv.csv


In [ ]:
import pandas as pd
import numpy as np
from scipy.interpolate import CubicSpline
from scipy.fft import fft
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('1_f.csv', header=None)
rr_intervals = df[0].values

In [ ]:
# I. Временной анализ ВСР
time_cumsum = np.cumsum(rr_intervals) / 1000
time_points = np.insert(time_cumsum, 0, 0)[:-1]

In [ ]:

# Статистические показатели
MxDMn = np.max(rr_intervals) - np.min(rr_intervals)
RRNN = np.mean(rr_intervals)
SDNN = np.std(rr_intervals, ddof=1)
RMSSD = np.sqrt(np.mean(np.diff(rr_intervals)**2))
CV = (SDNN / RRNN) * 100
HR = 60000 / RRNN  # ЧСС в уд/мин

# Расчет Амо (амплитуда моды) - процент от наиболее часто встречающегося интервала
hist, bin_edges = np.histogram(rr_intervals, bins=50)
AMo = (np.max(hist) / len(rr_intervals)) * 100

# Индекс напряжения (SI)
Mo_index = np.argmax(hist)
Mo = (bin_edges[Mo_index] + bin_edges[Mo_index + 1]) / 2
SI = AMo / (2 * Mo * MxDMn / 1000)  # Формула: АМо/(2*Мо*MxDMn)


In [ ]:
# II. Спектральный анализ
dt = 0.25
cs = CubicSpline(time_points, rr_intervals)
time_resampled = np.arange(0, time_points[-1], dt)
rr_resampled = cs(time_resampled)

N = len(rr_resampled)
window = np.hanning(N)
rr_windowed = rr_resampled * window

fft_result = fft(rr_windowed)
fft_magnitude = np.abs(fft_result)
psd = (fft_magnitude ** 2) / N

N_half = N // 2
psd_single = psd[:N_half] * 2
frequencies = np.fft.fftfreq(N, dt)[:N_half]

# Частотные диапазоны
HF_range = (0.15, 0.4)
LF_range = (0.04, 0.15)
VLF_range = (0.015, 0.04)

def calculate_power_in_band(frequencies, psd, freq_range):
    mask = (frequencies >= freq_range[0]) & (frequencies <= freq_range[1])
    power = np.trapz(psd[mask], frequencies[mask])
    return power

HF_power = calculate_power_in_band(frequencies, psd_single, HF_range)
LF_power = calculate_power_in_band(frequencies, psd_single, LF_range)
VLF_power = calculate_power_in_band(frequencies, psd_single, VLF_range)

TP = HF_power + LF_power + VLF_power
HF_percent = (HF_power / TP) * 100
LF_percent = (LF_power / TP) * 100
VLF_percent = (VLF_power / TP) * 100

/tmp/ipython-input-1267873116.py:26: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  power = np.trapz(psd[mask], frequencies[mask])


In [ ]:
pars_scores = {}

# А. Суммарный эффект регуляции (ЧСС)
if HR >= 90:
    pars_scores['A_HR'] = 2
    hr_description = f"Выраженная тахикардия (ЧСС={HR:.1f})"
elif HR >= 80:
    pars_scores['A_HR'] = 1
    hr_description = f"Умеренная тахикардия (ЧСС={HR:.1f})"
elif HR >= 60:
    pars_scores['A_HR'] = 0
    hr_description = f"Нормокардия (ЧСС={HR:.1f})"
elif HR >= 51:
    pars_scores['A_HR'] = -1
    hr_description = f"Умеренная брадикардия (ЧСС={HR:.1f})"
else:
    pars_scores['A_HR'] = -2
    hr_description = f"Выраженная брадикардия (ЧСС={HR:.1f})"

# Б. Функция автоматизма
if MxDMn <= 60 and CV <= 2:
    pars_scores['B_Automatism'] = 2
    auto_description = f"Стабильный ритм (MxDMn={MxDMn:.1f}, CV={CV:.2f})"
elif 60 < MxDMn < 150 and 2.0 < CV <= 4.0:
    pars_scores['B_Automatism'] = 1
    auto_description = f"Умеренная стабильность (MxDMn={MxDMn:.1f}, CV={CV:.2f})"
elif 150 <= MxDMn <= 300:
    pars_scores['B_Automatism'] = 0
    auto_description = f"Нарушений не выявлено (MxDMn={MxDMn:.1f})"
elif 300 < MxDMn <= 500:
    pars_scores['B_Automatism'] = -1
    auto_description = f"Умеренная аритмия (MxDMn={MxDMn:.1f})"
else:
    pars_scores['B_Automatism'] = -2
    auto_description = f"Выраженная аритмия (MxDMn={MxDMn:.1f})"

In [ ]:
# В. Вегетативный гомеостаз
if MxDMn <= 60 and AMo > 80:
    pars_scores['C_Homeostasis'] = 2
    homeo_description = f"Выраженное преобладание симпатики (MxDMn={MxDMn:.1f}, АМо={AMo:.1f})"
elif 60 < MxDMn < 150 and 51 <= AMo <= 80:
    pars_scores['C_Homeostasis'] = 1
    homeo_description = f"Умеренное преобладание симпатики (MxDMn={MxDMn:.1f}, АМо={AMo:.1f})"
elif 150 <= MxDMn <= 300 and 30 <= AMo <= 50:
    pars_scores['C_Homeostasis'] = 0
    homeo_description = f"Равновесие ВНС (MxDMn={MxDMn:.1f}, АМо={AMo:.1f})"
elif 300 < MxDMn <= 500 and 20 <= AMo < 30:
    pars_scores['C_Homeostasis'] = -1
    homeo_description = f"Умеренное преобладание парасимпатики (MxDMn={MxDMn:.1f}, АМо={AMo:.1f})"
else:
    pars_scores['C_Homeostasis'] = -2
    homeo_description = f"Выраженное преобладание парасимпатики (MxDMn={MxDMn:.1f}, АМо={AMo:.1f})"

# Г. Устойчивость регуляции
# Проверка на переходные процессы (если CV<15 и SI<15 - нестабильность)
if CV < 15 and SI < 15:
    pars_scores['D_Stability'] = 0  # "ложь" - нестабильность связана с переходными процессами
    stab_description = f"Нестабильность (переходные процессы) (CV={CV:.2f}, SI={SI:.2f})"
else:
    pars_scores['D_Stability'] = 0  # Нормальная устойчивость
    stab_description = f"Устойчивая регуляция (CV={CV:.2f}, SI={SI:.2f})"

In [ ]:
# Д1. Вазомоторный центр (LF%)
if LF_percent >= 55:
    pars_scores['D1_Vasomotor'] = 2
    vaso_description = f"Выраженное усиление вазомоторного центра (LF={LF_percent:.1f}%)"
elif 40 <= LF_percent < 55:
    pars_scores['D1_Vasomotor'] = 1
    vaso_description = f"Умеренное усиление вазомоторного центра (LF={LF_percent:.1f}%)"
elif 20 <= LF_percent < 40:
    pars_scores['D1_Vasomotor'] = 0
    vaso_description = f"Нормальная активность вазомоторного центра (LF={LF_percent:.1f}%)"
elif 11 <= LF_percent < 20:
    pars_scores['D1_Vasomotor'] = -1
    vaso_description = f"Умеренное ослабление вазомоторного центра (LF={LF_percent:.1f}%)"
else:
    pars_scores['D1_Vasomotor'] = -2
    vaso_description = f"Выраженное ослабление вазомоторного центра (LF={LF_percent:.1f}%)"

# Д2. Симпатический сердечно-сосудистый центр (VLF%)
if VLF_percent >= 60:
    pars_scores['D2_Sympathetic'] = 2
    symp_description = f"Выраженное усиление симпатического центра (VLF={VLF_percent:.1f}%)"
elif 45 <= VLF_percent < 60:
    pars_scores['D2_Sympathetic'] = 1
    symp_description = f"Умеренное усиление симпатического центра (VLF={VLF_percent:.1f}%)"
elif 25 <= VLF_percent < 45:
    pars_scores['D2_Sympathetic'] = 0
    symp_description = f"Нормальная активность симпатического центра (VLF={VLF_percent:.1f}%)"
elif 16 <= VLF_percent < 25:
    pars_scores['D2_Sympathetic'] = -1
    symp_description = f"Умеренное ослабление симпатического центра (VLF={VLF_percent:.1f}%)"
else:
    pars_scores['D2_Sympathetic'] = -2
    symp_description = f"Выраженное ослабление симпатического центра (VLF={VLF_percent:.1f}%)"

In [ ]:

print("\n" + "="*70)
print("ИСХОДНЫЕ ПОКАЗАТЕЛИ ВСР")
print("="*70)
print(f"ЧСС (HR):               {HR:.2f} уд/мин")
print(f"RRNN:                   {RRNN:.2f} мс")
print(f"SDNN:                   {SDNN:.2f} мс")
print(f"RMSSD:                  {RMSSD:.2f} мс")
print(f"MxDMn:                  {MxDMn:.2f} мс")
print(f"CV:                     {CV:.2f} %")
print(f"АМо:                    {AMo:.2f} %")
print(f"SI (индекс напряжения): {SI:.2f}")
print(f"\nСпектральные показатели:")
print(f"HF:                     {HF_percent:.2f} %")
print(f"LF:                     {LF_percent:.2f} %")
print(f"VLF:                    {VLF_percent:.2f} %")

print("\n" + "="*70)
print("РАСЧЕТ КОМПОНЕНТОВ ПАРС")
print("="*70)
print(f"\nА. Суммарный эффект регуляции (ЧСС)")
print(f"   Балл: {pars_scores['A_HR']:+2d} | {hr_description}")

print(f"\nБ. Функция автоматизма")
print(f"   Балл: {pars_scores['B_Automatism']:+2d} | {auto_description}")

print(f"\nВ. Вегетативный гомеостаз")
print(f"   Балл: {pars_scores['C_Homeostasis']:+2d} | {homeo_description}")

print(f"\nГ. Устойчивость регуляции")
print(f"   Балл: {pars_scores['D_Stability']:+2d} | {stab_description}")

print(f"\nД1. Вазомоторный центр")
print(f"   Балл: {pars_scores['D1_Vasomotor']:+2d} | {vaso_description}")

print(f"\nД2. Симпатический сердечно-сосудистый центр")
print(f"   Балл: {pars_scores['D2_Sympathetic']:+2d} | {symp_description}")




ИСХОДНЫЕ ПОКАЗАТЕЛИ ВСР
ЧСС (HR):               80.56 уд/мин
RRNN:                   744.80 мс
SDNN:                   38.70 мс
RMSSD:                  37.40 мс
MxDMn:                  229.00 мс
CV:                     5.20 %
АМо:                    6.33 %
SI (индекс напряжения): 0.02

Спектральные показатели:
HF:                     42.43 %
LF:                     43.15 %
VLF:                    14.42 %

РАСЧЕТ КОМПОНЕНТОВ ПАРС

А. Суммарный эффект регуляции (ЧСС)
   Балл: +1 | Умеренная тахикардия (ЧСС=80.6)

Б. Функция автоматизма
   Балл: +0 | Нарушений не выявлено (MxDMn=229.0)

В. Вегетативный гомеостаз
   Балл: -2 | Выраженное преобладание парасимпатики (MxDMn=229.0, АМо=6.3)

Г. Устойчивость регуляции
   Балл: +0 | Нестабильность (переходные процессы) (CV=5.20, SI=0.02)

Д1. Вазомоторный центр
   Балл: +1 | Умеренное усиление вазомоторного центра (LF=43.2%)

Д2. Симпатический сердечно-сосудистый центр
   Балл: -2 | Выраженное ослабление симпатического центра (VLF=14.4%)
